In [19]:
import json
import re
from pathlib import Path
from typing import List, Dict, Any, Optional
import pandas as pd
import fitz  # PyMuPDF

# ============================================================
# CONFIGURATION
# ============================================================
DOC_CODE = "Advisory Guidelines on Key Concepts in the PDPA 17 May 2022"
INPUT_JSON = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\{DOC_CODE}\hybrid_auto\{DOC_CODE}_content_list_corrected.json"
INPUT_PDF  = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\data privacy\{DOC_CODE}.pdf"
OUTPUT_CSV = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\graphrag_csv\{DOC_CODE}.csv"

# Block types to EXCLUDE
EXCLUDE_TYPES = {"image", "header", "footer", "page_number"}


# ============================================================
# Step 1: Extract text from a single block
# ============================================================
def extract_block_text(block: Dict[str, Any]) -> str:
    """Extract text content from a block based on its type."""
    typ = block.get("type", "")

    if typ == "text":
        return (block.get("text") or "").strip()

    elif typ == "table":
        return (block.get("table_body") or "").strip()

    elif typ == "list":
        items = block.get("list_items") or []
        return "\n".join(str(item).strip() for item in items if str(item).strip())

    elif typ == "code":
        return (block.get("code_body") or "").strip()

    elif typ == "equation":
        return (block.get("text") or "").strip()

    elif typ == "page_footnote":
        return (block.get("text") or "").strip()

    return (block.get("text") or "").strip()


# ============================================================
# Step 2: Extract PDF metadata
# ============================================================
def extract_pdf_metadata(pdf_path: str) -> Dict[str, Any]:
    """Extract metadata from PDF using fitz."""
    try:
        pdf = fitz.open(pdf_path)
        md = pdf.metadata or {}
        pdf.close()
        return {
            "doc_title": Path(pdf_path).stem or None,   # 用檔案名稱作為 doc_title
            "author":  md.get("author") or None,
            "version": md.get("version") or None,
        }
    except Exception as e:
        print(f"Warning: Could not read PDF metadata: {e}")
        return {"doc_title": Path(pdf_path).stem, "author": None, "version": None}


# ============================================================
# Step 3: Aggregate sections by text_level = 1
# ============================================================

def normalize_heading(text: str) -> str:
    """Normalize heading text for deduplication:
    - Remove trailing dots, ellipsis, page numbers
    - Collapse whitespace
    - Lowercase
    """
    text = re.sub(r'[\.…]+\s*\d*\s*$', '', text)  # trailing dots + page nums
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def aggregate_sections(content_list: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Aggregate content by text_level=1 headings.
    Uses last_occurrence to skip TOC duplicate headings:
    - TOC headings appear early (small index)
    - Body headings appear later (large index) → last occurrence = authoritative
    """

    # Pass 1: Find last occurrence index of each heading text
    last_occurrence: Dict[str, int] = {}
    for i, block in enumerate(content_list):
        if block.get("text_level") == 1:
            heading_text = normalize_heading(extract_block_text(block))
            if heading_text:
                last_occurrence[heading_text] = i  # overwrite → last wins

    print(f'  Unique headings found: {len(last_occurrence)}')

    # Pass 2: Aggregate sections, skipping TOC duplicate headings
    sections = []
    current_section_heading = None
    current_parts = []
    preamble_parts = []           # ← 收集第一個 heading 之前的內容

    for i, block in enumerate(content_list):
        typ = block.get("type", "")
        is_heading = block.get("text_level") == 1

        if typ in EXCLUDE_TYPES:
            continue

        if is_heading:
            heading_text = normalize_heading(extract_block_text(block))

            if last_occurrence.get(heading_text) != i:
                continue

            # Save preamble as first section (only once)
            if current_section_heading is None and preamble_parts:
                sections.append({
                    "section_title": "_preamble",
                    "text": "\n\n".join(p for p in preamble_parts if p),
                })

            # Save previous section
            if current_section_heading is not None:
                sections.append({
                    "section_title": current_section_heading,
                    "text": "\n\n".join(p for p in current_parts if p),
                })

            current_section_heading = extract_block_text(block)
            current_parts = [current_section_heading]

        else:
            text = extract_block_text(block)
            if text.strip():
                if current_section_heading is None:
                    preamble_parts.append(text)    # ← heading 前的內容存到 preamble
                else:
                    current_parts.append(text)

    # Last section
    if current_section_heading is not None:
        sections.append({
            "section_title": current_section_heading,
            "text": "\n\n".join(p for p in current_parts if p),
        })

    return sections


# ============================================================
# Step 4: Main execution
# ============================================================

# Load content list
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    content_list = json.load(f)
print(f"Loaded {len(content_list)} blocks from {INPUT_JSON}")

# Extract PDF metadata
metadata = extract_pdf_metadata(INPUT_PDF)
print(f"Metadata: {metadata}")

# Aggregate sections
sections = aggregate_sections(content_list)
print(f"Found {len(sections)} sections")

# Build DataFrame
df = pd.DataFrame(sections)
df["doc_title"] = metadata["doc_title"]
df["author"]    = metadata["author"]
df["version"]   = metadata["version"]

# Reorder columns
df = df[["text", "doc_title", "version", "author", "section_title"]]

before = len(df)
df = df[df["text"].str.strip() != df["section_title"].str.strip()]
# Ensure output directory exists
Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)

# Save CSV
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"✅ Saved to: {OUTPUT_CSV}")

display(df.head())

Loaded 1460 blocks from C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\Advisory Guidelines on Key Concepts in the PDPA 17 May 2022\hybrid_auto\Advisory Guidelines on Key Concepts in the PDPA 17 May 2022_content_list_corrected.json
Metadata: {'doc_title': 'Advisory Guidelines on Key Concepts in the PDPA 17 May 2022', 'author': None, 'version': None}
  Unique headings found: 5
Found 6 sections
✅ Saved to: C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\graphrag_csv\Advisory Guidelines on Key Concepts in the PDPA 17 May 2022.csv


,text,doc_title,version,author,section_title
0,ADVISORY GUIDELINES ON KEYCONCEPTS IN THE PERS...,Advisory Guidelines on Key Concepts in the PDP...,None,None,_preamble
1,PART I: INTRODUCTION AND OVERVIEW\n\n1 Introdu...,Advisory Guidelines on Key Concepts in the PDP...,None,None,PART I: INTRODUCTION AND OVERVIEW
2,PART II: IMPORTANT TERMS USED IN THE PDPA\n\n3...,Advisory Guidelines on Key Concepts in the PDP...,None,None,PART II: IMPORTANT TERMS USED IN THE PDPA
3,PART III: THE DATA PROTECTION PROVISIONS\n\n10...,Advisory Guidelines on Key Concepts in the PDP...,None,None,PART III: THE DATA PROTECTION PROVISIONS
4,PART IV: OFFENCES AFFECTING PERSONAL DATA AND ...,Advisory Guidelines on Key Concepts in the PDP...,None,None,PART IV: OFFENCES AFFECTING PERSONAL DATA AND ...


In [20]:
import pandas as pd

csv_path = OUTPUT_CSV

df = pd.read_csv(csv_path)
print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.iloc[0]['text'])         # 顯示內容
print(len(df.iloc[0]['text']))    # 顯示字數

Total rows: 6
Columns: ['text', 'doc_title', 'version', 'author', 'section_title']
ADVISORY GUIDELINES ON KEYCONCEPTS IN THE PERSONAL DATAPROTECTION ACT

Issued 23 September 2013

Revised 16 May 2022

TABLE OF CONTENTS

PART I: INTRODUCTION AND OVERVIEW....

Introduction...
Overview of the PDPA ... .8

PART II: IMPORTANT TERMS USED IN THE PDPA ........... .10

Definitions and related matters... .10
Individuals ... .11
Personal data.... ..12

When is data considered “personal data”?. .. 12
Truth and accuracy of personal data... . 16
Personal data relating to more than one individual . . 16
Excluded personal data . 1 7
Business contact information.. 17
Derived personal data .. 19
Personal data of deceased individuals . . 19
Control, not ownership, of personal data . . 20

6 Organisations...... ..22

Excluded organisations.. . 22
Individuals acting in a personal or domestic capacity.. . 23
Individuals acting as employees .. . 23
Public agencies .. . 24
Data intermediaries .. 24
Obligation